This notebook implements all analysis for the yeast dataset from Supplementary Figure 4F of [Vaishnav et al, 2022](https://www.nature.com/articles/s41586-022-04506-6). 

The imported dataset is a cleaned and reduced version from [CodeOcean](https://codeocean.com/capsule/8020974/tree/v1).

Specifically, it contains:
```
a. ∼3,929 sequences for yeast cultivated in synthetic defined medium
b. YFP fluorescence measurements, that we use as the target variable for all models
c. gene_ID tags for sequences that contain point mutations of the same promoter
d. group_ID tags that we assigned randomly to assess the impact of sequence diversity on model training
```

## 0. Mount Drive and set the main path

In [ ]:
#mount to drive
from google.colab import drive
try:
  drive.mount('/content/drive', force_remount=True)
except:
  drive._mount('/content/drive', force_remount=True)

# construct the path to use
import os
 
# get the current working directory
dir_path = os.path.dirname(os.path.realpath('__file__'))

# loop to crawl over your Drive and construct the correct path
for root, dirs, files in os.walk(dir_path):
    for file in files:
 
        # do not change the extension *.ipynb*
        if file.endswith('1_kmers.ipynb'):
          # check that we're using the correct parent directory
          if f'{root}/{file}'[-24:] == '/seq2yield/1_kmers.ipynb':
            # create path to the import directory
            drive_dir = f"{f'{root}/{file}'[:-13]}to_import/"

drive_dir

## 1. Import libraries and get data



In [ ]:
import os
import re
import time
import pickle
import random
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, drive_dir)

import utils, kMers, yeast_mod

# load .csv into pandas dataframe
pd.options.display.max_rows = 10
raw_data = pd.read_csv(f'{drive_dir}yeast_data.csv', index_col= 0).dropna().reset_index(drop = True)

## 2. Dimensionality reduction

### 2.1 Build a corpus for the dataset

**Note**: you may define a subsample of the dataset based on the number of sequences desired by varying the *num* argument

In [ ]:
# increase num to use more sequences
# change series to use different subsamples of the dataset
# change *k* to split sequences into k-mers of different lengths

num, k = raw_data.shape[0], 4

temp = raw_data[:num].copy()
temp = kMers.get_X_seqs(seq2prot=temp, k=k, organism='yeast')
corpus = [' '.join(list(temp.words)[item]) for item in range(temp.shape[0])]

# get # of counts per k-mer
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

DATA = cv.fit_transform(corpus).toarray()

### 2.2 UMAP (Fig 5A)

**Note**: you may notice an error about potential dependency conflicts because *pip*'s dependency resolver does not take into account all packages installed. The code block **will run** as intended.

In [ ]:
try:
  import umap.plot
except:
  !pip -q install umap-learn[plot]
  import umap.plot

# Run UMAP
mapper = umap.UMAP(densmap      = False, #set to *True* to strictly preserve local structure
                   n_neighbors  = 35,    #change the number of points to consider as neighbors 
                   min_dist     = 1,
                   n_components = 2,
                   low_memory   = True
                   ).fit(DATA)

# plot the 2D projection
fig = plt.figure(figsize = (10, 10))
sns.scatterplot(mapper.embedding_[:,0], 
                mapper.embedding_[:,1], 
                hue     = raw_data.native_gene, 
                palette = 'coolwarm', 
                alpha   = 0.2, 
                s = 6, 
                linewidths = 0.01, 
                legend = False
                ).plot();

## 3. Model training

### 3.1 Generate or load sets

In [ ]:
#generate train::validation::test sets
dev_set, test_set = yeast_mod.sample_genes(df_in = raw_data)
train_set, val_set = yeast_mod.sample_genes(df_in = dev_set)

# one-hot sequences of each set and organize in a dictionary
oh_dict = {f'{set_type}':yeast_mod.one_hot_lst(elem) for set_type, elem in zip(['train', 'val', 'test'], [train_set.sequence.values.tolist(), val_set.sequence.values.tolist(), test_set.sequence.values.tolist()])}

### 3.2 Model hyperparameter optimization

Below, I provide the output of a grid search with 10-fold cross-validation for Random Forests (used in the paper) and MLPs (not used in the paper).

RF: {'bootstrap': True, 'max_depth': 30, 'min_samples_leaf': 3, 'min_samples_split': 4, 'n_estimators': 50}

MLP: {'activation': 'tanh', 'hidden_layer_sizes': (250, 250, 250), 'max_iter': 200}


Define hyperparameter search spaces for *non-deep* regressors

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV

# define non-deep regressors and hyperparameters to search
regs = ['rf', 'mlp']
embeddings = ['one-hot', \
              #'1-mers', \
              #'3-mers', '4-mers', '5-mers', '6-mers',\
              #'3-count', '4-count','5-count','6-count',\
              #'features', 'mixed'
              ]

param_grid_rf = {
    'bootstrap': [True],
    'max_depth': [30, 50, 100],
    #'max_features': [2, 3],
    'min_samples_leaf': [3, 4],
    'min_samples_split': [2, 4, 6],
    'n_estimators': [25, 35, 50, 100]
}

param_grid_mlp = {
    'activation'         : ['logistic', 'relu', 'tanh'],
    'hidden_layer_sizes' : [
        (100, 100), (100, 100, 100), (200, 200),
        (200, 200, 200), (250, 250), (250, 250, 250)
    ],
    'max_iter'           : [100, 200, 250] 
}

Run the grid_seach

In [ ]:
# specify number of folds
cv_opt = 5

# lists to store predictions
rf_lst, mlp_lst= [], []
 
for embd_method in embeddings:

    # define a regressor params list
    for reg in regs:
      if reg == 'rf':
        # create a base RF model
        rf = RandomForestRegressor()
        # instantiate the grid search model
        grid_search_rf = GridSearchCV(estimator = rf, param_grid = param_grid_rf, 
                                      cv = cv_opt, n_jobs = -1, verbose = 2)
        grid_search_rf.fit(oh_dict['val'], val_set.protein.values.tolist())
        rf_lst.append(grid_search_rf.best_params_)

      elif reg == 'mlp':
        # create a base MLP model
        mlp = MLPRegressor()
        # instantiate the grid search model
        grid_search_mlp = GridSearchCV(estimator = mlp, param_grid = param_grid_mlp, 
                                      cv = cv_opt, n_jobs = -1, verbose = 2)
        grid_search_mlp.fit(oh_dict['val'], val_set.protein.values.tolist()) 
        mlp_lst.append(grid_search_mlp.best_params_)
 
      print(rf_lst, mlp_lst)

### 3.3 Train RF models on splits of the full dataset (not included in the paper)

In [ ]:
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split

train_size = 0.75

# simple train-test split
X_train, _, y_train, _ = train_test_split(oh_dict['train'], 
                                          train_set.protein.values.tolist(),
                                          test_size = 1-train_size)

X_test, y_test = oh_dict['test'], test_set.protein.values.tolist()

# init regressor obj
rf = RandomForestRegressor(n_estimators = 50,     
                           max_depth = 30,
                           min_samples_leaf = 3,    
                           min_samples_split = 4
                          )

# fit
rf.fit(X_train, y_train)

# get R2 
R2 = r2_score(y_test, rf.predict(X_test))

print(f'Trained on {len(X_train)} sequences\nTested on {len(X_test)} sequences\nR2 for the heldout test set is {R2}')


# scatterplot 
plt.subplots(figsize=(7,7))
sns.scatterplot(x = rf.predict(X_test), 
                y = y_test)

lims = (0, 100)
plt.ylim(lims)
plt.xlim(lims)
plt.xlabel('predicted')
plt.ylabel('measured')

## 4. Group-2-Group generalization for Random Forests (not included in the paper)

### 4.1 Generate groups of sequences based on gene_ID tags

In [ ]:
# create groups of clusters
no_groups = 12
clusters_per_group = int(199/no_groups) #total gene IDs divided by the number of groups we want

# create a list of all unique genes in the dataset
genes_lst = list(raw_data.native_gene.unique())

# initialize dictionaries
group_dict, group_test_dict = {}, {}

for group_itr in range(no_groups):
  try:
    sample_genes = random.sample(genes_lst, clusters_per_group)
  except:
    sample_genes = genes_lst.copy()
  genes_lst = [gene_id for gene_id in genes_lst if gene_id not in sample_genes]

  # drop group_ID column, since we are generating new ones
  group_dict[f'group_{group_itr + 1}'] = [train_set.drop(columns=['group_ID']).loc[train_set.native_gene == sample_id] for sample_id in sample_genes]
  group_test_dict[f'group_{group_itr + 1}'] = [test_set.drop(columns=['group_ID']).loc[test_set.native_gene == sample_id] for sample_id in sample_genes]

for group_itr in group_dict.keys():
  temp = group_dict[group_itr][0]
  temp_test = group_test_dict[group_itr][0]

  for elem, elem_test in zip(group_dict[group_itr], group_test_dict[group_itr]):
    temp = pd.concat((temp, elem)).reset_index(drop=True)
    temp_test = pd.concat((temp_test, elem_test)).reset_index(drop=True)
 
  group_dict[group_itr], group_test_dict[group_itr] = temp, temp_test

### 4.2 In- and cross-group testing of trained models

One-hot encode train and test groups

In [ ]:
work_oh_lst, test_oh_lst = [], []
for elem_w, elem_t in zip(group_dict, group_test_dict):
  seqs_oh_lst = yeast_mod.one_hot_lst(group_dict[elem_w].sequence.values.tolist())
  work_oh_lst.append(seqs_oh_lst)
  seqs_oh_lst_test = yeast_mod.one_hot_lst(group_test_dict[elem_t].sequence.values.tolist())
  test_oh_lst.append(seqs_oh_lst_test)

Run the loop

In [ ]:
train_size = 0.75 #change as desired to train on different data sizes per group

# cross-group training-testing loop
results_dict = {}
for group_itr in range(no_groups):

  # simple train-test split
  X_train, _, y_train, _ = train_test_split(work_oh_lst[group_itr], 
                                            group_dict[f'group_{group_itr + 1}'].protein.values.tolist(),
                                            test_size = 1-train_size)

  # init regressor obj
  rf = RandomForestRegressor(n_estimators = 50,     
                            max_depth = 30,
                            min_samples_leaf = 3,    
                            min_samples_split = 4
                            )

  # fit
  rf.fit(X_train, y_train)

  results_lst = []
  for test_group_itr in range(no_groups):
    X_test, y_test = test_oh_lst[test_group_itr], group_test_dict[f'group_{test_group_itr + 1}'].protein.values.tolist()

    # get R2 
    R2 = r2_score(y_test, rf.predict(X_test))
    results_lst.append(R2)
  
  results_dict[f'group_{group_itr + 1}'] = results_lst

# plot
sns.heatmap(pd.DataFrame.from_dict(results_dict),
            vmin=0,
            vmax=1,
            center = 0.5,
            cmap = 'Greens'
            )

## 5. Impact of sequence diversity on model generalization

**Note**: it is imperative that you execute **4** above for this block to work

### 5.1 Generate groups for 5 models

In [ ]:
iter_id = 1 

C_total = 400

in_, bw_ = yeast_mod.get_diversity(max_no_series = 12,
                                   mut_idxs = [itr+1 for itr in range(12)], 
                                   generate_new = True)

exp_dict = yeast_mod.diversity_sets(C = C_total,
                                    group_total = no_groups,
                                    group_dict = group_dict,
                                    in_series = in_,
                                    work_oh_lst = work_oh_lst,
                                    )

for exp in exp_dict.keys():
  print(f"{exp} contains {len(exp_dict[exp]['train'][0])} training sequences")

### 5.2 Training loop: 5 models

In [ ]:
results_dict = {}
for exp in exp_dict.keys():

  # simple train-test split
  X_train, y_train = exp_dict[exp]['train'][0], exp_dict[exp]['train'][1]

  # init regressor obj
  rf = RandomForestRegressor(n_estimators = 50,     
                             max_depth = 30,
                             min_samples_leaf = 3,    
                             min_samples_split = 4
                             )

  # fit
  rf.fit(X_train, y_train)

  results_lst = []
  for test_group_itr in range(no_groups):
    X_test, y_test = test_oh_lst[test_group_itr], group_test_dict[f'group_{test_group_itr + 1}'].protein.values.tolist()

    # get R2 
    R2 = r2_score(y_test, rf.predict(X_test))
    results_lst.append(R2)
  
  results_dict[exp] = results_lst

### 5.3 Plot results (Figs 5B & S13; insets)

In [ ]:
# get order of mutational series
idx_lst = in_[-1].copy()
idx_lst.extend(bw_[-1])

# dataframe with results from 27 models
res_df = pd.DataFrame.from_dict(results_dict)
res_df.index = [idx for idx in range(1, no_groups+1)]
res_df = res_df.loc[idx_lst]

# plot
sns.heatmap(res_df.T,
            vmin = 0,
            vmax = 1,
            center = 0.5,
            cmap=sns.cubehelix_palette(start=2, rot=0, dark=0.05, light=.98, reverse=False, as_cmap=True)
)  


### 5.4 Prediction accuracy across groups included in the training data (fig not included in the paper).

In [ ]:
nn, idx = [], 1
for col in res_df.columns:
  temp_av = np.mean(res_df[col].to_list()[:idx*2])
  nn.append(temp_av)
  idx+=1
  
plt.bar([itr for itr in range(1, res_df.shape[1]+1)], nn)
plt.ylabel('R2')
plt.xlabel('model')
plt.ylim((0.0, 1))

### 5.5 Diversity of training sequences
Counts the occurance of unique overlapping 5-mers across all sequences of each training set, and quantified as $1/\sum_{i=1}^{100}c_i$, where $c_i$ is the count of the top $i$ 5-mers. Specifically: 

```
- change k (->int) to test different k-mer lengths; default is 5-mers
- change *top_N* of top unique *k*-mers to generate a barplot sorted by increasing diversity (not included in the paper)

```

In [ ]:
exp_dict_df = {}
for exp in exp_dict.keys():
  exp_dict_df[exp] = pd.DataFrame({'Sequence' : exp_dict[exp]['train'][2],
                                   'protein'  : exp_dict[exp]['train'][1],
                                   'oh_seq'   : exp_dict[exp]['train'][0]}
                                  )
  
# rename to improve handling
exp_freq_dict, kmer = dict(), 5
for exp in exp_dict_df.keys():
  exp_dict_df[exp]['seed'] = [1 for itr in range(exp_dict_df[exp].shape[0])]
  exp_freq_dict[f'kFreq_{exp}'] = kMers.kmer_profiles(1, {'seed_1' : exp_dict_df[exp]}, kmer)

# create pandas dataframe for downstream processing and plotting
nnn = pd.DataFrame({'kmers'  : exp_freq_dict['kFreq_exp_1'].keys(),
                    'ID'     : [id for id in range(4**kmer)],
                    'counts' : [vl for vl in exp_freq_dict['kFreq_exp_1'].values()],
                    'exp'    : ['0' for exp_n in range(4**kmer)]
                    })

for itr, dat in enumerate(list(exp_freq_dict.keys())[1:]):
  nnn = pd.concat((nnn, pd.DataFrame({'kmers'  : exp_freq_dict['kFreq_exp_1'].keys(),
                                      'ID'     : [id for id in range(4**kmer)],
                                      'counts' : [vl for vl in exp_freq_dict[dat].values()],
                                      'exp'    : [f'{itr+1}' for exp_n in range(4**kmer)]})), axis=0).reset_index(drop=True)

new, new_sort = pd.DataFrame(), pd.DataFrame()
for itr in range(len(exp_dict)):
  new[f'exp_{itr}'] = nnn[nnn.exp == f'{itr}'].counts.reset_index(drop=True)

sort_based = 'all'
if sort_based == 'exp_0':
  new_sort = new.sort_values(by=sort_based,ascending=False).reset_index(drop=True)
else:
  for col in new:
    new_sort[col] = new[col].sort_values(ascending=False, ignore_index=True) 

# plot
top_N = 100 #default value of top 100 5-mers, as shown in Fig 5
plt.bar([itr for itr in range(1, res_df.shape[1]+1)], new_sort.loc[:top_N].sum().apply(lambda x : 1/x).values)
plt.ylabel('diversity')
plt.xlabel('model')

### 5.6 Compute mean R2 across *in-* and *cross-*group heldout test sets, and generate bubble plot (Figs 5B & S13)

In [ ]:
cr, idx = [], 1
for col in res_df.columns:
  temp_av = np.mean(res_df[col].to_list())
  cr.append(temp_av)
  idx+=1
  
nn = pd.DataFrame({'diversity' : (new_sort.loc[:top_N].sum().apply(lambda x : 1/x).values),
                   'sequences/series' : [int(C_total/no_sr) for no_sr in range(2, no_groups, 2)],
                   'accuracy' : cr})
nn['accuracy'] = nn.accuracy.apply(lambda x: 0 if x<0 else x)


sns.relplot(x="diversity", 
            y="sequences/series", 
            hue="accuracy", 
            size="accuracy",
            sizes=(20, 200), 
            alpha=.85, 
            palette="coolwarm",
            height = 5, 
            aspect = 1.5,
            data=nn)

plt.ylim((0, int(C_total/2) + 25))
plt.xlim((10e-5, 12.5e-5))
plt.savefig(f'yeast_divbubbles_group12_{iter_id}.svg')